# Vector addition: your first CUDA kernel

A CUDA kernel is a function that thousands of threads run at once. Each thread works out **which element it owns** from its block and thread coordinates, then does a tiny piece of work. Vector addition is the "hello world" of this model:

$$\text{out}_i = a_i + b_i, \qquad i = 0, \dots, n-1$$

Write a Numba CUDA kernel and a host function `vector_add(a, b)`.

## Inputs
- `a`, `b`: 1-D NumPy arrays of dtype `float32` with the same length `n` (`n` may be 0).

## What the host function must do
1. Copy `a` and `b` to the GPU (`cuda.to_device`) and allocate the output there (`cuda.device_array`).
2. Launch the kernel with **128 threads per block** and just enough blocks to cover `n`: $\lceil n / 128 \rceil$.
3. Copy the result back and return it as a `float32` NumPy array of shape `(n,)`.

## Indexing
The global index of a thread in a 1-D launch is

$$i = \text{blockIdx.x} \cdot \text{blockDim.x} + \text{threadIdx.x}$$

which `cuda.grid(1)` computes for you. The last block is usually only partly used, so threads with $i \ge n$ must do nothing.

## Edge case
A launch with zero blocks is an error. If `n == 0`, return an empty `float32` array without launching.

## Runtime
Runs on a GPU in Colab (**Runtime → Change runtime type → T4 GPU**). The tests here run on Numba's CUDA simulator, which executes every thread on the CPU, so sizes are small.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
!pip -q install "numba-cuda[cu12]"

In [ ]:
import numpy as np
from numba import cuda

TPB = 128  # threads per block

def vector_add(a, b):
    """a, b: float32 (n,). Returns a + b as float32 (n,)."""
    n = a.shape[0]
    out = np.zeros(n, dtype=np.float32)
    # TODO: copy a and b to the device, launch add_kernel[blocks, TPB](...), copy the result back
    return out

@cuda.jit
def add_kernel(a, b, out):
    # TODO: compute this thread's global index, guard against i >= n, write out[i] = a[i] + b[i]
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [1.0, 2.0, 3.0], "dtype": "float32"}, {"__nd__": [10.0, 20.0, 30.0], "dtype": "float32"}], "e": {"__nd__": [11.0, 22.0, 33.0], "dtype": "float32"}}, {"a": [{"__nd__": [-1.5, 0.0, 2.25, 4.0], "dtype": "float32"}, {"__nd__": [1.5, -3.0, 0.75, -4.0], "dtype": "float32"}], "e": {"__nd__": [0.0, -3.0, 3.0, 0.0], "dtype": "float32"}}, {"a": [{"__nd__": [], "dtype": "float32"}, {"__nd__": [], "dtype": "float32"}], "e": {"__nd__": [], "dtype": "float32"}}, {"a": [{"__nd__": [7.0], "dtype": "float32"}, {"__nd__": [-2.0], "dtype": "float32"}], "e": {"__nd__": [5.0], "dtype": "float32"}}, {"a": [{"__nd__": [0.3455841839313507, 0.8216181397438049, 0.3304370641708374, -1.3031572103500366, 0.9053558707237244, 0.4463745653629303, -0.5369532108306885, 0.581118106842041, 0.3645724058151245, 0.2941325008869171, 0.028422242030501366, 0.5467129945755005, -0.7364540696144104, -0.16290995478630066, -0.4821193218231201, 0.5988461971282959, 0.03972210735082626, -0.2924567461013794, -0.7819084525108337, -0.2571922540664673, 0.008142180740833282, -0.27560290694236755, 1.2940638065338135, 1.0067243576049805, -2.711162567138672, -1.8890132904052734, -0.17477209866046906, -0.422190397977829, 0.21364299952983856, 0.21732193231582642, 2.1178388595581055, -1.11202073097229, -0.37760502099990845, 2.042771577835083, 0.6467030048370361, 0.6630633473396301, -0.5140063762664795, -1.6480752229690552, 0.1674647480249405, 0.10901408642530441, -1.2273520231246948, -0.6832266449928284, -0.07204367965459824, -0.9447516202926636, -0.09826996922492981, 0.09548302739858627, 0.03558623790740967, -0.5062916874885559, 0.5937480926513672, 0.8911669254302979, 0.3208483159542084, -0.8182302117347717, 0.7316522598266602, -0.5014400482177734, 0.8791606426239014, -1.0717873573303223, 0.9144672155380249, -0.020063454285264015, -1.2487488985061646, -0.3138994574546814, 0.054102279245853424, 0.27279132604599, -0.9821881055831909, -1.1073729991912842, 0.19958452880382538, -0.4667496085166931, 0.235505610704422, 0.7595195174217224, -1.6487873792648315, 0.25438812375068665, 1.2246469259262085, -0.29752683639526367, -0.8108145594596863, 0.7522438168525696, 0.2534465193748474, 0.8958830833435059, -0.34521570801734924, -1.4818183183670044, -0.11001076549291611, -0.4458281397819519, 0.7753238081932068, 0.1936328411102295, -1.6308492422103882, -1.1951631307601929, 0.8837890625, 0.6797650456428528, -0.6402433514595032, -0.0010487965773791075, 0.4455735683441162, 0.4684043228626251, 0.8762422204017639, 0.25648564100265503, -0.09482833743095398, -0.25884807109832764, 1.055742859840393, -2.250854253768921, -0.13865531980991364, 0.03300010412931442, -1.4253489971160889, 0.3328136205673218, -0.6512809991836548, 0.8624448180198669, -0.12559208273887634, 0.6691532135009766, 1.2188435792922974, 0.38292959332466125, -0.8757211565971375, -1.514318585395813, 1.7533841133117676, -0.11129219084978104, -0.6885649561882019, 0.1442570835351944, -0.1914113312959671, 0.8521422743797302, 0.03392818197607994, 0.013749583624303341, -0.714579701423645, 0.4695681035518646, -1.0338667631149292, 0.6658894419670105, 1.523937463760376, -1.5246859788894653, -2.466229200363159, 0.6168787479400635, 2.5478978157043457, -1.000924825668335, -1.2506957054138184, 0.5889689326286316], "dtype": "float32"}, {"__nd__": [-0.840721607208252, -0.506025493144989, -0.348117470741272, 0.5320020914077759, -0.40530237555503845, 0.277882844209671, -0.17653325200080872, -0.8446711301803589, -0.31982624530792236, -0.95039963722229, 0.006514986045658588, -1.1238662004470825, -1.0928943157196045, 1.4569618701934814, -0.05318421870470047, -0.053902026265859604, 0.5115364193916321, -0.4208570122718811, -0.22853536903858185, 0.42514872550964355, 0.28241583704948425, -1.1592967510223389, 0.8333426117897034, -0.5904349684715271, -1.0560789108276367, -0.9004750847816467, -0.3905453383922577, 1.6273002624511719, -1.1755359172821045, 0.16007588803768158, -2.137824296951294, -0.001566933118738234, 0.8995664119720459, -0.2366633266210556, -0.6293548941612244, 0.23151107132434845, 0.7001517415046692, 0.663657546043396, 1.9724738597869873, 0.2091674655675888, -0.5924100875854492, -0.12597918510437012, -0.07249854505062103, 0.10873738676309586, -0.030027812346816063, 0.1739659458398819, -1.6708500385284424, 0.8296289443969727, -0.5747392773628235, -1.1731586456298828, 0.6377511620521545, 1.3173259496688843, 0.4930281639099121, 0.16115930676460266, -0.9322202801704407, 2.8715672492980957, 0.8802586197853088, -1.1392946243286133, -0.7796379327774048, 0.08697924762964249, -1.5547311305999756, 0.16863040626049042, -0.459071546792984, 1.2262705564498901, 0.9621546864509583, -2.711285352706909, 0.04170258715748787, -1.6174675226211548, 1.109637975692749, 0.1681058704853058, 0.5484054684638977, -1.065124750137329, 1.8284302949905396, 2.020073413848877, -1.064771056175232, 0.3728151321411133, -0.673302412033081, -0.02356993593275547, -1.265636920928955, 1.8671455383300781, -0.9691795110702515, -0.29608380794525146, 0.5014829039573669, -0.6475606560707092, -0.23931242525577545, -0.5636398196220398, -0.13346076011657715, -1.170542597770691, -0.4379880726337433, -0.2068929225206375, -0.33372601866722107, 0.05668995529413223, -0.2931022047996521, 0.7532113790512085, -0.3231959342956543, -0.13664959371089935, -0.6647813320159912, -0.5265148282051086, -1.2644927501678467, 0.5187849402427673, -1.1425180435180664, -0.7458564043045044, 0.3592446446418762, 0.4025733768939972, -0.4001147449016571, -2.01926589012146, 0.4205133020877838, 0.2595634460449219, -1.4123811721801758, 0.7703220844268799, -0.7010998129844666, -1.1261881589889526, 0.09573071449995041, -0.1784704327583313, 0.20262400805950165, -1.6057480573654175, 1.812230110168457, -0.6026586294174194, -1.5396592617034912, 0.6188421845436096, -0.354804128408432, 0.32485848665237427, -0.33960843086242676, -0.05974036082625389, 0.2457728385925293, -0.7466529011726379, 0.678739607334137, -0.46990010142326355], "dtype": "float32"}], "e": {"__nd__": [-0.49513742327690125, 0.3155926465988159, -0.01768040657043457, -0.7711551189422607, 0.5000535249710083, 0.7242574095726013, -0.7134864330291748, -0.26355302333831787, 0.04474616050720215, -0.6562671661376953, 0.03493722900748253, -0.577153205871582, -1.8293483257293701, 1.2940518856048584, -0.53530353307724, 0.544944167137146, 0.5512585043907166, -0.7133137583732605, -1.0104438066482544, 0.16795647144317627, 0.29055801033973694, -1.4348996877670288, 2.127406358718872, 0.41628938913345337, -3.7672414779663086, -2.7894883155822754, -0.5653174519538879, 1.2051098346710205, -0.9618929028511047, 0.3773978352546692, -0.019985437393188477, -1.1135876178741455, 0.5219613909721375, 1.8061082363128662, 0.017348110675811768, 0.8945744037628174, 0.1861453652381897, -0.9844176769256592, 2.1399385929107666, 0.3181815445423126, -1.819762110710144, -0.8092058300971985, -0.14454221725463867, -0.8360142111778259, -0.12829777598381042, 0.2694489657878876, -1.6352638006210327, 0.32333725690841675, 0.0190088152885437, -0.28199172019958496, 0.9585994482040405, 0.49909573793411255, 1.2246804237365723, -0.3402807414531708, -0.05305963754653931, 1.7997798919677734, 1.7947258949279785, -1.159358024597168, -2.0283868312835693, -0.2269202172756195, -1.5006288290023804, 0.4414217472076416, -1.4412596225738525, 0.11889755725860596, 1.1617392301559448, -3.178035020828247, 0.27720820903778076, -0.8579480051994324, -0.5391494035720825, 0.42249399423599243, 1.773052453994751, -1.3626515865325928, 1.017615795135498, 2.7723171710968018, -0.8113245368003845, 1.2686982154846191, -1.018518090248108, -1.5053882598876953, -1.3756476640701294, 1.4213173389434814, -0.19385570287704468, -0.10245096683502197, -1.129366397857666, -1.8427238464355469, 0.6444766521453857, 0.11612522602081299, -0.7737041115760803, -1.1715914011001587, 0.007585495710372925, 0.2615113854408264, 0.5425162315368652, 0.31317558884620667, -0.3879305422306061, 0.49436330795288086, 0.7325469255447388, -2.3875038623809814, -0.8034366369247437, -0.4935147166252136, -2.6898417472839355, 0.8515985608100891, -1.7937990427017212, 0.11658841371536255, 0.23365256190299988, 1.0717265605926514, 0.8187288045883179, -1.636336326599121, -0.45520785450935364, -1.2547551393508911, 0.3410029411315918, 0.6590299010276794, -1.3896647691726685, -0.9819310903549194, -0.0956806167960167, 0.6736718416213989, 0.2365521937608719, -1.5919984579086304, 1.097650408744812, -0.1330905258655548, -2.573525905609131, 1.2847316265106201, 1.1691333055496216, -1.1998274326324463, -2.805837631225586, 0.5571383833885193, 2.793670654296875, -1.7475776672363281, -0.5719560980796814, 0.11906883120536804], "dtype": "float32"}}, {"a": [{"__nd__": [-0.8696871399879456, 0.0770324245095253, 0.4450412690639496, -0.22907933592796326, -0.8625198006629944, 0.6197855472564697, -1.760328769683838, -1.0308641195297241, 0.039522890001535416, -1.3610594272613525, 0.027994263917207718, -0.05486311763525009, 0.8987398147583008, -0.9147903323173523, -0.6259065270423889, 0.3331816792488098, -2.457563638687134, 3.1000423431396484, -0.6986507177352905, -0.729835033416748, 0.861127495765686, -0.039831843227148056, -1.7794286012649536, 0.626927375793457, 0.8553778529167175, -0.4499462842941284, -0.28160035610198975, 0.48598459362983704, -0.9087802767753601, 0.43838855624198914, 0.19929851591587067, -0.674932599067688, -1.3921018838882446, -0.2256058305501938, -0.8754222393035889, 1.0014102458953857, 0.1440853625535965, 0.7820845246315002, 0.13462193310260773, 0.2629011273384094, -0.7829989194869995, 0.6680474281311035, 1.784698247909546, -0.3096875548362732, -0.592774510383606, -0.1578367054462433, -0.48128029704093933, -0.7014793157577515, 0.13819363713264465, -0.2909175455570221, 1.4388736486434937, 0.00020164294983260334, 0.32391196489334106, 0.9520218968391418, -0.3007558584213257, 1.4367365837097168, -0.6326941847801208, -0.8083276748657227, -0.36626702547073364, -0.11471739411354065, -1.4013181924819946, -0.03509478271007538, -1.667486548423767, 1.3921383619308472, -0.08099695295095444, -0.641957700252533, -0.9083381295204163, -0.384430468082428, -0.22307947278022766, -1.0445109605789185, -0.9197950959205627, -0.18717500567436218, -0.5207837224006653, 0.9392130970954895, 1.13787043094635, 0.016021203249692917, 0.4735995829105377, -1.335188388824463, 0.637417733669281, -0.030583469197154045, 0.4846683144569397, 1.600356101989746, -2.280857801437378, 0.26094818115234375, -1.0991190671920776, 0.5921967029571533, -1.3133134841918945, -0.4953995645046234, 0.20273375511169434, 0.6135093569755554, 0.07479409128427505, -0.7928309440612793, -0.5535322427749634, 0.8845028877258301, -0.005499615799635649, -1.6838144063949585, 0.843662679195404, 0.41624847054481506, 0.8734155297279358, -0.33662745356559753, 0.8281565308570862, -1.0610649585723877, 0.5699998140335083, -0.49038031697273254, 0.674351692199707, 1.0056440830230713, -0.7359901666641235, -0.051229480654001236, 0.03895462676882744, 1.1896648406982422, 0.7105581164360046, -1.2192775011062622, 0.45760825276374817, 0.7450892329216003, 2.1238059997558594, -1.6791491508483887, -0.536352276802063, 1.3333711624145508, -1.3550703525543213, -1.1994606256484985, 0.5170822143554688, 1.0184086561203003, -0.6686804890632629, 0.540127158164978, 0.11695527285337448, 1.5187489986419678, -0.0015184397343546152, 0.9902473092079163, -0.903117835521698], "dtype": "float32"}, {"__nd__": [-0.1848788559436798, -0.09670445322990417, 1.1391079425811768, 0.5796130299568176, -0.7517531514167786, 0.6819677352905273, 0.7706312537193298, -0.11164572089910507, -0.2576623260974884, -0.19380325078964233, -1.6949924230575562, 0.1887430101633072, 0.23457792401313782, -0.8655285239219666, 0.74244624376297, -1.3729668855667114, -0.5518128275871277, -0.47521907091140747, 1.987509846687317, -1.5991840362548828, 0.5626302361488342, 0.942123293876648, 0.3777255713939667, 1.1818079948425293, -1.0027376413345337, -2.2802586555480957, 0.7668178677558899, -1.1958281993865967, -0.32639986276626587, -1.178846001625061, 1.0523418188095093, 0.8651502132415771, -0.7200636863708496, 0.9034918546676636, 0.12127725780010223, -0.1393888294696808, 0.05741666257381439, -0.2027493566274643, 0.6150052547454834, 0.3103475570678711, -0.3494671583175659, 1.0062557458877563, -0.6114755868911743, 0.28693363070487976, 0.42215538024902344, 1.4788070917129517, -0.5054413676261902, 1.7430824041366577, 0.17616495490074158, -0.19397690892219543, -0.6724727749824524, 0.5862933993339539, 0.04893524572253227, -1.103609323501587, -1.1278871297836304, -0.5555192232131958, -0.676006019115448, 1.0903023481369019, 1.3755383491516113, 0.8685595989227295, 0.35786888003349304, -0.4282382130622864, 0.054483212530612946, 0.8847070336341858, 2.130972146987915, 0.9128991961479187, -0.28041213750839233, 0.03802507743239403, -0.4825448989868164, -0.7823985815048218, -0.18424199521541595, 0.19561469554901123, 1.8359016180038452, 0.052032470703125, 1.3573580980300903, 1.7674890756607056, 0.07880786806344986, 1.6077989339828491, 0.7149354219436646, -0.41753530502319336, 0.2651800215244293, 0.02287781797349453, -0.2323390245437622, -0.19878123700618744, 0.14420931041240692, 0.4341811537742615, -0.884881854057312, -0.009387661702930927, -1.522355079650879, 0.2607955038547516, 0.6235994696617126, 0.16395483911037445, 0.28585314750671387, 0.5899431109428406, -0.6651008725166321, -0.23904885351657867, 0.5109220147132874, 1.0018857717514038, 0.3949213922023773, 2.552859306335449, -0.09136056900024414, 0.9996604323387146, 1.2667276859283447, -0.13376984000205994, -0.8205128908157349, -1.182602882385254, 0.1622769981622696, 1.1102919578552246, 0.271668404340744, 0.17285607755184174, -0.3792138695716858, 0.5610297322273254, -2.135831117630005, 0.23237325251102448, 0.02812630869448185, -1.3703402280807495, 2.175597906112671, -1.3874132633209229, -1.077520489692688, -1.2008631229400635, 1.110367774963379, -0.888084888458252, 0.6686564087867737, 0.5875101685523987, 0.2596704065799713, -1.3075789213180542, -0.6121063828468323, 1.6731150150299072, -1.2907544374465942], "dtype": "float32"}], "e": {"__nd__": [-1.0545660257339478, -0.019672028720378876, 1.5841492414474487, 0.35053369402885437, -1.614272952079773, 1.301753282546997, -0.9896975159645081, -1.1425098180770874, -0.2181394398212433, -1.5548627376556396, -1.6669981479644775, 0.1338798999786377, 1.1333177089691162, -1.7803188562393188, 0.11653971672058105, -1.0397851467132568, -3.0093765258789062, 2.6248233318328857, 1.2888591289520264, -2.329019069671631, 1.423757791519165, 0.902291476726532, -1.4017030000686646, 1.8087353706359863, -0.14735978841781616, -2.7302050590515137, 0.48521751165390015, -0.709843635559082, -1.235180139541626, -0.7404574155807495, 1.2516403198242188, 0.19021761417388916, -2.1121654510498047, 0.6778860092163086, -0.7541449666023254, 0.8620214462280273, 0.2015020251274109, 0.5793351531028748, 0.7496271729469299, 0.5732486844062805, -1.1324660778045654, 1.6743031740188599, 1.1732226610183716, -0.022753924131393433, -0.17061913013458252, 1.3209704160690308, -0.9867216348648071, 1.0416030883789062, 0.31435859203338623, -0.48489445447921753, 0.7664008736610413, 0.586495041847229, 0.37284719944000244, -0.15158742666244507, -1.428642988204956, 0.881217360496521, -1.3087002038955688, 0.2819746732711792, 1.0092713832855225, 0.7538422346115112, -1.0434492826461792, -0.46333301067352295, -1.6130033731460571, 2.2768454551696777, 2.0499751567840576, 0.27094149589538574, -1.1887502670288086, -0.34640538692474365, -0.7056243419647217, -1.8269095420837402, -1.1040370464324951, 0.008439689874649048, 1.3151178359985352, 0.9912455677986145, 2.4952285289764404, 1.7835103273391724, 0.552407443523407, 0.27261054515838623, 1.3523530960083008, -0.44811877608299255, 0.7498483657836914, 1.6232339143753052, -2.5131969451904297, 0.06216694414615631, -0.9549097418785095, 1.0263779163360596, -2.198195457458496, -0.5047872066497803, -1.3196213245391846, 0.8743048906326294, 0.6983935832977295, -0.6288760900497437, -0.2676790952682495, 1.4744460582733154, -0.6706004738807678, -1.922863245010376, 1.3545846939086914, 1.4181342124938965, 1.2683368921279907, 2.2162318229675293, 0.736795961856842, -0.061404526233673096, 1.836727499961853, -0.6241501569747925, -0.14616119861602783, -0.17695879936218262, -0.5737131834030151, 1.0590624809265137, 0.31062301993370056, 1.3625209331512451, 0.33134424686431885, -0.6582477688789368, -1.678222894668579, 0.9774624705314636, 2.1519322395324707, -3.0494894981384277, 1.639245629310608, -0.05404210090637207, -2.432590961456299, -2.4003238677978516, 1.6274499893188477, 0.13032376766204834, -2.4080276489257812e-05, 1.1276373863220215, 0.3766256868839264, 0.21117007732391357, -0.6136248111724854, 2.6633622646331787, -2.1938722133636475], "dtype": "float32"}}, {"a": [{"__nd__": [-0.831654965877533, -0.16224652528762817, 0.8089900612831116, 0.251639187335968, 0.7420441508293152, -1.0672942399978638, 0.9447765946388245, 0.5690287947654724, -1.5945645570755005, 1.5399161577224731, 2.292397975921631, -0.7685773372650146, 0.055652815848588943, 1.3974225521087646, -1.4812402725219727, -1.9898602962493896, -1.2969441413879395, -0.5670667886734009, -0.5784422159194946, 0.6073471903800964, 0.26853635907173157, -1.246077299118042, 0.5673799514770508, 1.8740813732147217, 1.1970542669296265, 0.9911074638366699, 0.02017792873084545, 0.9836313724517822, -0.9661328196525574, 0.7510146498680115, -0.08724842220544815, 1.1309057474136353, 0.46601054072380066, -1.0898797512054443, 0.12799295783042908, 1.2182214260101318, -1.1292338371276855, -0.5591399073600769, -0.7686730027198792, -1.4956459999084473, 0.9612682461738586, 1.3109972476959229, 0.8000432252883911, 0.24257031083106995, -0.05203138291835785, 0.23938030004501343, -0.7315559387207031, 0.8938456177711487, 1.0454922914505005, 0.9325435161590576, -0.5309482216835022, 0.07840434461832047, -0.16578081250190735, 1.7885748147964478, 0.1796962022781372, -1.8820708990097046, 0.39850565791130066, 1.8822662830352783, 0.6851757168769836, 0.879479706287384, 0.03606126829981804, -1.9713895320892334, -1.8102577924728394, -1.2459477186203003, -0.1268320381641388, 0.309998482465744, 0.6894141435623169, -0.3404580354690552, 0.9569292664527893, -0.2797487676143646, -0.704489529132843, 0.8516417145729065, -0.9144449830055237, -2.7290849685668945, -1.0599567890167236, 0.09415954351425171, -3.0801777839660645, -0.3571380376815796, -0.3320903480052948, -1.4264559745788574, -1.4825034141540527, -0.4624619483947754, -0.5471001863479614, 1.259918451309204, 0.37486305832862854, -1.5808554887771606, -0.8546514511108398, 0.705141544342041, 1.8959767818450928, 0.3864994943141937, 0.3107125461101532, 1.8607524633407593, -0.022671444341540337, -0.3091009557247162, -1.4132487773895264, -0.5130161046981812, 2.178712844848633, -1.4217413663864136, 0.011191707104444504, -1.4083274602890015, 0.12847575545310974, 0.8876472115516663, -0.24364931881427765, 0.7303472757339478, 0.7208438515663147, 0.4470248222351074, 1.7162480354309082, 0.7788375616073608, -0.30495956540107727, -0.680823564529419, -0.8453063368797302, 0.47586745023727417, -0.32372182607650757, 2.7291669845581055, 1.8424476385116577, -0.21452400088310242, -0.32913124561309814, 1.6901766061782837, -1.8833597898483276, -0.45155104994773865, 0.9507569670677185, -0.9120975732803345, -0.47682997584342957, -0.35114115476608276, 0.6608085036277771, 0.13752688467502594, 0.40208742022514343, 0.5952837467193604, 0.0299382284283638, 1.1105661392211914, -0.7289588451385498, -1.4361214637756348, -1.7768718004226685, -1.6210098266601562, -0.5505157113075256, 1.0300469398498535, 0.1741667538881302, -0.5985466241836548, -1.0698555707931519, -0.5288856625556946, -0.4036663770675659, -0.5650994181632996, -1.4325443506240845, -0.4758078157901764, 1.0171360969543457, -2.1746089458465576, -0.24060867726802826, -2.1213483810424805, 0.1830476075410843, -0.7474893927574158, -2.0856876373291016, 0.1007021889090538, 1.817405343055725, 0.10889153182506561, 1.161791443824768, -0.07150439918041229, -2.157590866088867, 0.49048349261283875, -0.4182793200016022, -1.4620848894119263, 0.7848366498947144, 0.28685757517814636, -0.5254096388816833, 0.8680694103240967, -0.8720719814300537, 0.6434323191642761, -0.9404604434967041, 0.5339547395706177, -1.1007479429244995, 1.4891881942749023, 0.25430750846862793, 1.432875156402588, -0.8224459886550903, -0.47610238194465637, 0.8396648168563843, -2.7437171936035156, -1.0667908191680908, 1.0331768989562988, 0.46510931849479675, 0.9787693619728088, -0.3097608983516693, 0.563999354839325, -0.730844259262085, -1.3719604015350342, -1.69479501247406, -0.5517095923423767, -0.548941433429718, -0.6293774247169495, -3.548804998397827, -1.9986035823822021, -0.5326324701309204, 0.2870166301727295, -0.5787926316261292, -0.8768972158432007, -2.3084733486175537, 2.0214991569519043, -1.1011112928390503, 1.6833966970443726, 1.5606858730316162, 0.27392905950546265, -0.5726094245910645, 0.13221552968025208, -1.153041958808899, 0.260730117559433, 0.002952147740870714, 0.5319682955741882, 0.9504368305206299, 1.73283851146698, -0.3883158266544342, 1.0433883666992188, -0.7916696071624756, -0.8945649266242981, -0.6180564165115356, -0.9819656610488892, 0.9802643060684204, 0.923241138458252, 1.2723116874694824, -0.02530032955110073, 0.2743816077709198, -0.5784270167350769, -1.1509612798690796, 0.4470181465148926, 0.2683731019496918, -1.10701322555542, 0.585332453250885, -1.741727352142334, -0.17899848520755768, -0.6303138136863708, -0.5292761921882629, -1.0834577083587646, 0.9524940848350525, 0.5659481883049011, 1.1834203004837036, 0.13049735128879547, -1.2980536222457886, -0.37900152802467346, -1.1915178298950195, 0.44679516553878784, 1.1987059116363525, 0.3160904049873352, -0.6125209331512451, -0.3833271265029907, 0.19044999778270721, -1.460357427597046, 0.1346021443605423, 0.18349787592887878, -0.5602428913116455, -0.4340352714061737, 0.15384463965892792, -0.8382888436317444, -0.8404628038406372, 0.7599708437919617, 0.20337072014808655, 1.6586636304855347, -1.0079779624938965, -1.8009659051895142, 0.9187788963317871, 0.9341444373130798, -0.7621833086013794, -1.4928569793701172, -0.07219059020280838, -1.7779053449630737, -0.36896464228630066, -2.2109122276306152, -0.2679457664489746, -2.151350259780884, -0.26883959770202637, 1.4845112562179565, -0.3006952404975891, 0.7352385520935059, -0.6770767569541931, -0.9284830093383789, -1.5350784063339233, 1.0023084878921509, -0.09697972983121872, 1.5823183059692383, -1.2198184728622437, -0.2650924026966095, 0.036186788231134415, 1.3681716918945312, 1.7769346237182617, -0.6945933699607849, -1.1802839040756226, -1.7838454246520996, 1.288723111152649, -0.13599422574043274, -0.6974769234657288, -0.10144000500440598, -2.6089084148406982, -1.7766330242156982, 0.8838022947311401, 0.34403255581855774, 0.09807949513196945, -1.0536868572235107, 0.5353450775146484, 1.7259052991867065, -1.2772682905197144, -0.03816312924027443, -0.48818373680114746, 0.3076118230819702, -0.039425212889909744, -0.20061995089054108, -0.9358153343200684, -0.06958525627851486, -0.6682911515235901, -0.2003965824842453, -0.11751377582550049, 0.6002955436706543, -0.6363173127174377, -0.37742021679878235, 1.017966866493225, -0.272327184677124, -0.3159642219543457, 0.8102701306343079, 0.2185261845588684, 0.037242624908685684, 0.16619548201560974, -1.293020248413086, 0.408203661441803, 1.2737979888916016, 0.5299164652824402, -1.6061770915985107, 0.23526029288768768, -0.9527249336242676, -0.2733396291732788, 0.08390972018241882, -1.2370305061340332, 1.2921054363250732, -0.11166165769100189, 0.7504063844680786, 0.5934798717498779, 1.303356647491455, 1.1517683267593384, -1.513160228729248, 0.004810902755707502, -2.172405481338501, -0.4837769865989685, 0.6784226894378662, 0.8782966732978821, 1.2538259029388428, -0.41177764534950256, 0.28596407175064087, 0.06367363035678864, 1.2336868047714233, 1.555299997329712, -0.3915332853794098, 3.7516350746154785, -0.03604073449969292, -0.3988884389400482, 0.567840039730072, 0.7271924018859863, 0.5156142711639404, -0.6796766519546509, 1.443800926208496, 0.025823863223195076, 1.5104701519012451, 1.3665618896484375, -1.298855185508728, -1.0043131113052368, -1.024233102798462, 0.22707755863666534, 0.032114122062921524, 0.24757523834705353, -0.8327108025550842, 0.4071473777294159, 1.3851046562194824, -1.3535946607589722, -0.2137952446937561, 0.23100575804710388, -0.14705057442188263, -0.21412745118141174, 1.7522640228271484, 1.267067551612854, 0.8333949446678162, -0.7771995663642883, 2.225606679916382, 0.3459026515483856, -0.592975914478302, -0.3523772954940796, -0.5043286085128784, 2.106536626815796, 0.19118697941303253, 0.049319684505462646, -2.166612148284912, 0.7238102555274963, -1.0714960098266602, -1.1429566144943237, 0.5995576977729797, -0.8764085173606873, 0.8228349685668945, 1.1088930368423462, -1.8023948669433594, -0.14911003410816193, -1.586817979812622, -0.0749339833855629, 0.8991979956626892, -1.376827359199524, 1.7891597747802734, -0.3246411383152008, -1.1975678205490112, -0.1393173485994339, 0.7767747044563293, -1.3929332494735718, -0.01426596101373434, -1.3898124694824219, 1.1875461339950562, 0.13787288963794708, -0.4426848590373993, 1.0174864530563354, -1.3066112995147705, -0.41168680787086487, 1.8275185823440552, -0.1430663764476776, 1.3375608921051025, 0.1757359504699707, -0.719624936580658, -0.26622137427330017, 0.056934040039777756, -0.4300137162208557, -0.8447017073631287, -0.8023917078971863, -1.5044679641723633, 0.08910326659679413, 0.37208083271980286, 0.7272999286651611, -1.82758629322052, 0.7838677167892456, -0.03910607099533081, -1.4506160020828247, 0.07583510875701904, 0.7925947904586792, 0.5004207491874695, 0.7033272385597229, 0.9233344197273254, 0.8739572763442993, -0.05745004862546921, -2.1042306423187256, -0.7812315821647644, 0.3345961272716522, -0.4908728301525116, 0.6643822193145752, 0.7768180966377258, 0.027355611324310303, 1.859187126159668, -0.9955329298973083, 0.9110583066940308, -0.49258601665496826, -0.8004662990570068, 0.568815290927887, -0.6356137990951538, 0.8392890095710754, -1.1862937211990356, 0.015583934262394905, -0.8629299402236938, -0.2544780671596527, -0.39141133427619934, -1.147969365119934, 0.6362796425819397, -1.264243483543396, -0.9990895390510559, 0.8727511763572693, 0.9547403454780579, 0.6494452953338623, 0.3220405876636505, 2.26027250289917, -1.2274054288864136, -0.760988175868988, -1.011277437210083, 0.8814588189125061, 0.2710099518299103, 1.093985915184021, 0.9557623267173767, -0.24162495136260986, -0.8353966474533081, -1.008449912071228, 2.195673704147339, 0.951823353767395, 0.5775319933891296, -0.354330450296402, -0.21339817345142365, -0.3590921461582184, -0.3387848138809204, -2.41849946975708, -0.07619941979646683, -0.3971336781978607, 0.2749563157558441, 0.19483956694602966, 0.8387179970741272, -0.027649279683828354, 0.7824317812919617, -2.5754430294036865, -1.1156771183013916, 1.7114428281784058, -1.2335188388824463, -0.29330477118492126, 0.6926035284996033, -0.8967195749282837, 0.7083745002746582, -1.4422564506530762, -1.9275062084197998, -1.3434523344039917, -1.6678674221038818, -0.0318593755364418, -0.33193933963775635, 0.3142775893211365, -1.0498944520950317, 2.528757333755493, -0.1096208319067955, -0.0599173903465271, 0.4416601359844208, -0.25759974122047424, -0.645520806312561, 0.07179854065179825, 0.9047775864601135, 0.6517590284347534, 0.9280803799629211, -0.5777807235717773, -0.7883017063140869, 1.9911670684814453, -0.25188976526260376, 0.46569374203681946, 0.7259218096733093, 1.4085023403167725, -1.36322820186615, -0.49206921458244324, -0.4452933073043823, -0.31308069825172424, -2.4242351055145264, -0.10759609192609787, -2.9051733016967773, 0.2870934009552002, 0.5268477201461792, 0.5874223113059998, -1.2150816917419434, 0.9450477957725525, -0.8739072680473328, -1.2368271350860596, -0.9898289442062378, -0.8570382595062256, 0.16350944340229034, 0.8160107135772705, -2.110056161880493, -0.5492770671844482, 0.8939716219902039, 0.19619494676589966, 0.5017732977867126, -1.5046709775924683, 0.9487374424934387, -1.5689442157745361, 0.1900714635848999, 1.3612574338912964, 1.8793504238128662, 0.5115365386009216, 0.880969226360321, -0.6063316464424133, -0.4814441204071045, 1.1694233417510986, 0.001606382429599762, -0.12731382250785828, -1.4105581045150757, 0.15059097111225128, 0.64586341381073, -0.7941694855690002, -0.5070798397064209, 1.5804566144943237, 0.17826615273952484, -0.2885936498641968, -1.136982798576355, -0.619579017162323, -0.24999624490737915, 1.369127869606018, -0.6104946732521057, 1.7943357229232788, -0.9286878108978271, -0.3575868010520935, 0.8435436487197876, -0.24019493162631989, -0.8302620649337769, -0.1318434178829193, 0.4487124979496002, -1.7157342433929443, -0.2485363483428955, -0.10933662205934525, -0.4037477672100067, 0.7353560924530029, -0.5685521960258484, -0.36451485753059387, 0.03764258697628975, 0.7288491129875183, -0.2211626172065735, 1.5165972709655762, -0.907718300819397, 1.286514401435852, -0.857821524143219, -0.16306647658348083, 0.364046186208725, 2.054839849472046, 0.8958261609077454, -0.40586283802986145, 1.2991191148757935, -0.2137048989534378, -0.7914458513259888, 0.4486846923828125, 0.09447085112333298, 0.9536663889884949, -0.549738347530365, 0.38150349259376526, 0.6345493197441101, 0.27804556488990784, -0.028983531519770622, -0.6976183652877808, 0.903751790523529, 0.1372244954109192, 0.4963039755821228, -0.3507782518863678, 0.01301169116050005, -1.037227988243103, 0.08552529662847519, 0.7652103304862976, 0.09299328923225403, 0.8593928217887878, 0.3129104673862457, 1.3638566732406616, -0.19004963338375092, -1.6386302709579468, 0.43248823285102844, -0.1651894897222519, -0.9742934107780457, 0.6512653827667236, 0.26432913541793823, 0.4453386068344116, -0.3323759436607361, 0.197047159075737, 0.2543390393257141, 1.7746551036834717, -0.6839597821235657, -0.010494199581444263, 0.36385980248451233, -1.7247868776321411, -1.6029683351516724, 2.165818452835083, -1.7671189308166504, 0.5749291181564331, -0.8482254147529602, -0.7631601095199585, -1.9825248718261719, -0.9007385969161987, 0.7843785881996155, -1.3887428045272827, 0.5954268574714661, 0.31329774856567383, 2.0183022022247314, -0.4745335876941681, 1.2634140253067017, 0.8888689279556274, -0.28666841983795166, -1.6629559993743896, 1.5094404220581055, -1.3643889427185059, -0.47341594099998474, 0.8831765651702881, 1.1159555912017822, -0.8454467058181763, -0.03200168535113335, -0.25857844948768616, 1.181558609008789, 2.5046334266662598, -1.829886794090271, 0.9164229035377502, -2.858175277709961, 0.014066372066736221, -1.108784556388855, 1.9918187856674194, 0.1663685292005539, -0.7367717623710632, -0.4993128478527069, -0.5752957463264465, 0.1883116066455841, -0.15814846754074097, 0.6558855772018433, 1.5402610301971436, 0.7014120221138, -1.2260030508041382, -1.6238068342208862, -0.7554195523262024, 1.0532152652740479, -0.3350481390953064, -0.6675727963447571, 0.13828255236148834, -1.4155259132385254, -1.1393277645111084, 0.5103254318237305, -0.30315348505973816, -0.45323625206947327, 0.31763502955436707, -0.584871232509613, -0.6862910389900208, -0.45981016755104065, 0.6026206016540527, 0.059819936752319336, -1.194597601890564, -0.303056925535202, 0.6482239365577698, 1.870751976966858, -1.1423540115356445, 0.3385353088378906, -0.25874826312065125, 0.8255194425582886, 0.449554979801178, 1.4721769094467163, 0.5194509029388428, -0.5451133251190186, -0.5385931730270386, 0.40555012226104736, 0.3949431777000427, 2.397937297821045, 1.676332712173462, -1.1735783815383911, 0.3502247929573059, -1.330988883972168, -1.6556768417358398, 0.7435460090637207, 0.5517475605010986, -1.0385677814483643, 1.6436867713928223, -0.9775699377059937, -1.1677284240722656, 0.6526182293891907, 0.2209385484457016, -0.16050800681114197, -0.06706583499908447, 0.22380079329013824, -0.13957998156547546, -0.17250487208366394, -1.8959566354751587, -1.34196138381958, 0.6109495759010315, -0.7043008804321289, 0.5451932549476624, 1.2354638576507568, -0.35495081543922424, 0.28296977281570435, 2.0764763355255127, 0.6949356198310852, 1.8418738842010498, 1.5334070920944214, 0.7533603310585022, -0.11189375817775726, -0.841993510723114, -1.111698031425476, 0.48632892966270447, 0.46818336844444275, -0.17462220788002014, 0.8484663963317871, -0.6045371294021606, 1.110219120979309, 0.04000706598162651, -0.0363265760242939, -0.0002732969878707081, -0.09859127551317215, 2.488264322280884, -1.6115198135375977, 0.17722254991531372, 1.2441061735153198, 0.7175130248069763, -0.30208197236061096, 1.7401210069656372, -1.0684928894042969, -1.3062961101531982, 0.5680705904960632, -1.4544697999954224, -2.3779497146606445, 0.8828866481781006, 0.7613503336906433, 0.7322868704795837, -0.4874711334705353, 0.1829461306333542, -0.9653266072273254, 1.7343511581420898, 0.7910242676734924, 0.7422560453414917, -1.9499847888946533, 0.8073020577430725, 1.4939720630645752, 1.2162621021270752, 0.7891303300857544, -0.0009571690461598337, -0.34183284640312195, 0.6016417145729065, 0.3827693462371826, 0.23637452721595764, -0.3092247545719147, 1.6812565326690674, -1.1678433418273926, -0.022632455453276634, -0.5801808834075928, 1.0320430994033813, 1.4304815530776978, 1.244088888168335, 0.7554071545600891, -1.402601957321167, 2.081232786178589, 0.5166960954666138, -0.10846324265003204, 0.021500596776604652, -0.3395191431045532, 0.42388057708740234, 0.3101678192615509, -0.36010003089904785, -2.5698914527893066, -0.035121336579322815, 0.4768359661102295, 0.5261678695678711, -0.15346500277519226, 0.45331478118896484, 1.0283188819885254, -0.47025027871131897, 1.4662790298461914, -1.9551583528518677, -1.2610485553741455, -0.9051449298858643, -2.4444665908813477, -0.9997625946998596, 1.248371958732605, 1.241529107093811, 0.09735846519470215, 1.7210019826889038, 0.23949022591114044, -0.4978119432926178, 0.24163076281547546, -0.13714660704135895, 0.7359334826469421, -1.503050684928894, 0.9994149208068848, -0.4595267176628113, 0.8553230166435242, -1.5410401821136475, -0.41925865411758423, 0.07806321978569031, 0.9014526009559631, -1.2175310850143433, 1.6674442291259766, 0.48266348242759705, -0.5257754325866699, 1.2173020839691162, 0.102711021900177, 1.1582969427108765, 2.323147773742676, 0.768528401851654, -0.5266794562339783, -0.4257074296474457, 0.5825543403625488, 0.28790542483329773, -0.8330299854278564, -0.6681095361709595, 0.6657820343971252, 2.34669828414917, -2.6303672790527344, 0.7239161133766174, -0.2110471874475479, 0.7793647050857544, 0.8677213788032532, 0.05283808335661888, -1.0159735679626465, -1.3970519304275513, -1.8741786479949951, 1.2223455905914307, -0.20970237255096436, -0.540274977684021, -0.2777194082736969, -1.3394677639007568, 0.3046145737171173, 1.3187015056610107, 1.138460636138916, -1.5657991170883179, 1.3635119199752808, -0.7642368674278259, -1.9129523038864136, -1.229472279548645, -0.9665141105651855, -0.006947840563952923, -1.031975507736206, -1.510245442390442, -0.26449403166770935, 1.9239039421081543, -0.7023508548736572, 0.4642535150051117, 0.07994683086872101, 2.6580677032470703, -1.8982582092285156, 1.0953816175460815, -1.2813849449157715, -0.5924152731895447, -0.09668165445327759, 3.1860861778259277, -0.7630053162574768, -1.8888840675354004, 1.2973636388778687, -0.8392360806465149, -0.24295035004615784, -0.02404182218015194, 0.09698878228664398, -0.1701386272907257, 0.2677564322948456, 0.3873595893383026, 0.8539228439331055, -0.42723312973976135, -0.7487301230430603, -1.0635502338409424, 0.5927088856697083, -0.04629814624786377, 0.5532045960426331, 0.5381962656974792, 1.6858769655227661, 0.20750509202480316, 0.7523713111877441, -1.2025898694992065, -1.0803016424179077, -0.3117377460002899, -1.2167177200317383, 0.6895931959152222, -1.3330527544021606, 0.06838301569223404, -2.4580955505371094, -1.1047797203063965, -0.3390353322029114, -0.2590397000312805, -1.7916457653045654, 0.10866907984018326, 0.9748640060424805, 0.6482403874397278, -0.6585830450057983, -0.5872968435287476, -1.186300277709961, 0.6297879815101624, 1.441658854484558, -0.5259554982185364, -1.6978180408477783, 0.20000424981117249, 0.7077694535255432, -2.722123861312866, 0.1383560448884964, 2.1643869876861572, 0.8206475973129272, 0.5504587888717651, 0.40212127566337585, 1.0618988275527954, -0.9411672353744507, -0.8453378081321716, 0.4075693190097809, -0.6853293180465698, 0.25870242714881897, -1.9125624895095825, 0.7579546570777893, 0.8469713926315308, 0.04367790371179581, 0.16442294418811798, 1.620651125907898, -1.083094596862793, -0.9131566286087036, -1.093806266784668, -0.42473018169403076, -1.0111498832702637, 0.03704847767949104, -0.6507368087768555, -1.4491491317749023, -1.5501776933670044, 1.211657166481018, -0.06887881457805634, -0.2273378223180771, 0.026569586247205734, -0.0823621153831482, 0.7561129331588745, 0.5622590184211731, -0.5897077322006226, -0.8235270380973816, -1.0867208242416382, -1.251131296157837, 1.4021456241607666, -0.9165392518043518, -2.019045829772949, -0.8434844613075256, -0.788361668586731, 0.3298516273498535, -0.8859825730323792, 0.5347530841827393, 0.388526976108551, 1.266829490661621, 2.1669931411743164, 2.8489041328430176, 1.350834608078003, -1.1597002744674683, -0.40419939160346985, 0.24764613807201385, 2.068127393722534, 0.6766203045845032, 1.2033723592758179, -0.1739518791437149, 0.5902844667434692, 0.7181922793388367], "dtype": "float32"}, {"__nd__": [-0.4563084840774536, 0.7773279547691345, 1.0708105564117432, -0.15836574137210846, 1.0181465148925781, 1.6479547023773193, 0.0662999376654625, 0.0003221777733415365, -0.309617280960083, -0.6193094849586487, 0.15615114569664001, -0.41182684898376465, -0.311980664730072, -0.7734912633895874, -1.5360480546951294, -0.9372441172599792, 0.3289974331855774, -1.183138370513916, -0.647973895072937, 1.642160177230835, 1.1265157461166382, -0.6934160590171814, 0.6033794283866882, -0.6502859592437744, 1.627589225769043, -0.5856937170028687, 0.08171670138835907, 1.3848086595535278, -0.8278767466545105, 1.3045341968536377, -0.8174513578414917, 0.2381957471370697, -0.29561468958854675, -0.08850503712892532, 0.16009029746055603, -0.07434438914060593, -1.0173333883285522, 0.8665798306465149, 0.7205393314361572, 1.2898365259170532, 1.3849481344223022, -1.3439677953720093, 1.2855112552642822, 1.2977346181869507, -0.5322431921958923, 1.0374046564102173, 1.468231201171875, -0.5388355255126953, -1.3787593841552734, -0.3595365285873413, 2.7927069664001465, 0.23626045882701874, -1.2142094373703003, -0.25294727087020874, -0.9940522313117981, -0.3992806077003479, 0.0951441153883934, -2.5615971088409424, 0.3604007959365845, 1.4146696329116821, -0.1973688006401062, -0.38269492983818054, 1.093590497970581, -1.2666386365890503, -1.2458901405334473, -1.3905360698699951, -0.1573779582977295, 0.3940269649028778, -0.09057294577360153, 0.5497761964797974, 0.8307471871376038, 0.49658527970314026, -0.8086538314819336, -0.2680876851081848, 1.2482471466064453, -1.0466866493225098, -0.25342369079589844, -0.4172738194465637, -0.18008865416049957, -1.075668454170227, -1.3493335247039795, 2.5559639930725098, 0.945901095867157, -0.7126172184944153, -0.05896497517824173, 0.6767085194587708, -0.34366098046302795, 0.6165235638618469, -0.6152846813201904, -0.35852915048599243, -0.08476875722408295, -0.4574955105781555, -2.45282244682312, 0.021152084693312645, 0.26162201166152954, 0.4761964678764343, -0.9939849972724915, 0.34424737095832825, 0.21570058166980743, -1.7688179016113281, -2.201935291290283, 0.47647058963775635, -0.9541853070259094, 1.0022063255310059, 2.140371322631836, -0.5827268958091736, -0.6893947720527649, -0.011696125380694866, 0.6747466921806335, -1.1357991695404053, 1.245471715927124, 0.9912216067314148, 0.6611064672470093, -0.5398409962654114, 0.9163020849227905, 0.8572128415107727, -1.2009708881378174, 2.090184211730957, -0.30052733421325684, 0.3691859841346741, -0.5241554379463196, 0.38489678502082825, 1.160082459449768, -0.8073686361312866, -1.6554259061813354, -0.44998547434806824, -1.3060460090637207, -0.7585726976394653, -0.1264830082654953, -0.0641574040055275, 1.2907289266586304, -1.0147984027862549, 0.3527083992958069, -0.5075674653053284, 1.3650569915771484, 2.527698516845703, 0.303439736366272, 1.6741909980773926, -0.5302056074142456, -1.5994064807891846, -1.270398497581482, -1.579089879989624, -0.954749345779419, -0.24021869897842407, 1.1501034498214722, 0.699859082698822, 0.4637638330459595, -0.4237360656261444, -0.7219123244285583, 0.033264655619859695, 0.1376667320728302, 0.25621122121810913, 0.3715481162071228, -0.11761394888162613, -1.9246562719345093, -1.1152390241622925, -0.825614333152771, 1.696732759475708, -0.4272724688053131, 0.6750341057777405, 1.4642385244369507, 0.3333636522293091, -0.3927915692329407, -0.29322099685668945, -1.0223926305770874, 0.3065388798713684, 1.1280697584152222, 0.5698114037513733, 0.771317183971405, 0.7176809310913086, 2.0040230751037598, -0.6338130831718445, 0.15752385556697845, 0.7054760456085205, 1.2472662925720215, -2.427642345428467, -0.5223416090011597, -0.2084936648607254, -1.029420256614685, 0.8471343517303467, 0.2189895212650299, -0.39644962549209595, -1.6341204643249512, -1.3237723112106323, -0.9853346943855286, 0.45481202006340027, -1.2192609310150146, -2.069427967071533, -1.4525175094604492, -0.5919622182846069, 1.6732755899429321, 2.5019304752349854, -0.4813788831233978, 0.9438583850860596, 1.4012110233306885, 1.2845419645309448, -0.1167856752872467, -1.3020793199539185, -0.48830434679985046, -1.8619823455810547, 0.23863568902015686, 0.7774940133094788, -1.8929904699325562, -0.051819223910570145, 0.2923556864261627, 1.2493208646774292, 1.378623127937317, 0.8012739419937134, -0.6867340803146362, -0.3683476746082306, 1.20429265499115, -0.6059065461158752, -0.04224791377782822, 1.5608105659484863, 0.12788154184818268, -1.3745152950286865, -0.29558974504470825, 0.6393219828605652, -0.19033092260360718, 1.1613121032714844, 1.2820096015930176, -0.6508179306983948, -0.5036587119102478, -0.10430659353733063, -1.2017639875411987, 0.3778183162212372, -0.180227130651474, 0.7852035164833069, -0.6001034379005432, -0.8797667026519775, 1.8377834558486938, -0.8793261051177979, -0.5856227278709412, -0.7324876189231873, -0.3046237528324127, -0.9160462617874146, -0.04498622193932533, -0.7048763632774353, -0.41687512397766113, 0.36281800270080566, 0.701580822467804, 0.466969758272171, -0.736066460609436, 0.39716440439224243, -0.12046289443969727, -0.10632529109716415, -1.7223691940307617, 0.6713172197341919, -0.13960538804531097, 1.7062674760818481, 0.6401019096374512, -1.1165173053741455, -1.3721935749053955, 1.150978446006775, -0.8748712539672852, 0.6844912171363831, -0.3007708787918091, 2.103834629058838, 0.080193430185318, 1.5157731771469116, 1.3341472148895264, -0.4610562324523926, -0.32605162262916565, 1.4895120859146118, 0.00707817729562521, -0.8606792092323303, 1.2136573791503906, 2.5848464965820312, -0.023499388247728348, 1.418412446975708, -0.13985492289066315, 1.381151556968689, -0.2114691585302353, 1.3636295795440674, 0.5778641700744629, -0.14685556292533875, 0.27587100863456726, -0.5007154941558838, -1.0825896263122559, -1.0106388330459595, -0.008610596880316734, -1.0073091983795166, 0.4934638440608978, 0.5872507691383362, 0.5287472605705261, -0.43694475293159485, 0.11835166811943054, 0.037558384239673615, 1.3873541355133057, -0.7418610453605652, -0.3186655342578888, -0.166934996843338, -0.07552280277013779, -0.6816709041595459, -0.5605828762054443, 1.0851173400878906, -0.5692654848098755, 0.9872300624847412, 2.105889081954956, 1.349249005317688, -1.7379447221755981, 0.8446711301803589, 0.5495827198028564, 0.6156253218650818, -1.5560507774353027, 0.830219030380249, -1.019120693206787, 1.3645570278167725, 1.1414307355880737, 2.618600845336914, -0.31923529505729675, -1.4587770700454712, 1.1142841577529907, 0.9391270279884338, -0.3186103105545044, 1.4249008893966675, 0.7527135610580444, -1.225251317024231, -0.2889012396335602, -0.3836996555328369, -0.047463417053222656, 0.2541768550872803, -0.6339845061302185, -0.1358189731836319, 0.6513442993164062, -0.012387270107865334, -0.20923323929309845, -0.24376355111598969, -0.039485808461904526, -1.85512113571167, 0.22104096412658691, -0.9681957364082336, 0.1666085571050644, -1.411927342414856, 0.9827263951301575, 0.2974168360233307, -1.2909389734268188, 0.2977229356765747, -0.6992514729499817, 1.8621320724487305, 0.5519067049026489, -0.7357704043388367, 0.28648921847343445, 0.029280662536621094, 1.2606269121170044, 0.42016032338142395, 0.5776007771492004, -1.054145097732544, 0.8575826287269592, 2.159860134124756, 0.703392744064331, 0.027037162333726883, 0.23072504997253418, -0.8860154151916504, 0.25335270166397095, -1.031011939048767, 0.11177787184715271, -0.3203124403953552, -0.7628656029701233, 0.6486691236495972, -1.4213436841964722, 0.028216559439897537, -1.2564973831176758, 1.8992775678634644, 2.0443289279937744, 0.9158250093460083, 1.9667402505874634, 0.6624847054481506, 0.6115452647209167, -0.3565219044685364, -0.8269065022468567, 0.19622208178043365, -0.4673181176185608, -0.369951993227005, 0.7968809604644775, -0.060071010142564774, -1.4936186075210571, -0.14981390535831451, -0.6971672177314758, -0.22878164052963257, -1.0322670936584473, 1.2654510736465454, 1.9679852724075317, -1.0467257499694824, 0.7133212089538574, 0.6625272631645203, -1.0100986957550049, 0.16530150175094604, 1.4751231670379639, 0.9105765223503113, 1.6823347806930542, -1.1079763174057007, -0.684907853603363, 0.3522082269191742, 0.41434982419013977, 0.0834614560008049, -0.32882025837898254, -0.35335007309913635, -0.5514953136444092, 0.5062891244888306, -2.705317497253418, 1.653336524963379, -0.475048691034317, -1.3785929679870605, 0.5954136252403259, -1.3552217483520508, 0.43523111939430237, -0.353611022233963, 0.637843132019043, 0.835486114025116, 1.3398066759109497, -0.45129841566085815, -0.24807213246822357, 0.029863504692912102, -1.3734409809112549, 0.6336269378662109, 0.16870832443237305, -1.3013224601745605, 1.2491611242294312, 1.404166579246521, -1.2792757749557495, 0.6095300912857056, 1.2876684665679932, 0.8764917254447937, -0.9449921250343323, -1.3618921041488647, -0.08067025989294052, -0.46189603209495544, 0.025706058368086815, -0.06977847218513489, 0.5600060224533081, -0.5285683274269104, -0.13033953309059143, 2.999063491821289, -0.09943142533302307, 0.7662734985351562, 0.31756022572517395, 0.2464880794286728, -0.8841350078582764, 0.554287850856781, -0.02489212341606617, 0.8436720967292786, 2.076913833618164, 0.20986561477184296, 0.7863034605979919, -0.9989746809005737, 0.14108529686927795, 1.5753825902938843, -0.03627828136086464, -0.613615870475769, 0.2296043187379837, -0.16168460249900818, -0.21776486933231354, -0.34998658299446106, 2.0091657638549805, -0.84861820936203, -0.3906412422657013, -0.5482403635978699, 1.2621115446090698, -0.25697413086891174, 1.326125979423523, 0.18822000920772552, -0.0017454883782193065, -1.4392125606536865, 1.232831358909607, 0.5623341798782349, 0.4263381361961365, -0.09617636352777481, -0.39546868205070496, -1.2468876838684082, 0.9669301509857178, 0.2901197373867035, -0.3379421830177307, -0.6091782450675964, 0.1372336447238922, -0.06966158747673035, 1.6641099452972412, 0.7951828837394714, -0.574727475643158, -1.0841946601867676, -0.21445722877979279, -0.8900456428527832, 0.09774206578731537, 0.4260566234588623, 0.6586355566978455, 0.4390936493873596, -0.16686971485614777, 0.6246941685676575, -0.8642482161521912, -0.052823979407548904, 0.8386164307594299, 0.3147051930427551, -1.2439250946044922, -1.1273174285888672, -1.5866607427597046, 1.8330591917037964, -1.520209550857544, -0.10803203284740448, 1.8672457933425903, 1.108313798904419, 1.3488130569458008, -0.4669523239135742, 0.7714552283287048, 3.078183174133301, -0.003428217489272356, 0.4468066096305847, -0.28159603476524353, 0.02079235203564167, 0.40802547335624695, 1.4213756322860718, -0.8654747605323792, -1.393388271331787, 0.06109648942947388, 0.241029754281044, 1.4472237825393677, 0.7537670731544495, -0.5881070494651794, 0.6031578779220581, 1.1526423692703247, 0.3227618932723999, -0.303257554769516, 0.7131995558738708, -0.31848767399787903, -1.8310294151306152, 1.3667794466018677, -1.254354476928711, -0.2999145984649658, -0.2818320691585541, -0.30168095231056213, -0.25180351734161377, -1.4079707860946655, 0.5573773980140686, 0.058324720710515976, 0.06954293698072433, 1.101508378982544, 1.1286895275115967, 1.36752450466156, 1.87883460521698, 0.48127391934394836, -1.510216236114502, 0.24072955548763275, 0.040813155472278595, 0.09686562418937683, -0.5724860429763794, -0.8609741926193237, 0.8243263363838196, 0.9978535175323486, -1.0898874998092651, 0.567503035068512, -0.14109055697917938, -0.359906941652298, 0.7120957970619202, -0.023610174655914307, -0.5268955230712891, 0.082660973072052, -1.970350742340088, -0.5667799115180969, -0.41502684354782104, -1.3743740320205688, 0.19965383410453796, 0.2902805805206299, -0.5823673009872437, -0.15174199640750885, 0.690491795539856, 1.0761768817901611, -0.7446040511131287, 0.35357633233070374, 1.050766110420227, -0.017603622749447823, -0.7600975632667542, -0.7563065886497498, -0.3128760755062103, -1.0246715545654297, 2.020066976547241, -0.5635942816734314, 0.3837192952632904, 0.18796853721141815, 0.6409974694252014, -0.3821176588535309, -1.296480417251587, 1.639717698097229, -0.07751930505037308, 0.11699814349412918, 0.2003602236509323, 1.2616794109344482, 0.7690678834915161, 1.1740642786026, 0.8228328227996826, 0.15384669601917267, 0.25526878237724304, -0.7417651414871216, -1.9371566772460938, -0.23776812851428986, -1.0713318586349487, 2.588639497756958, 1.5604217052459717, 1.1721856594085693, -0.5491815805435181, 2.060317277908325, -1.702633261680603, -1.2068897485733032, -0.7793229818344116, -0.9564529657363892, 0.13407188653945923, 0.19540713727474213, 1.9115464687347412, 0.9989392757415771, 1.2224456071853638, 0.4200282692909241, -0.14565883576869965, -0.6848222017288208, -1.5075562000274658, -0.868141770362854, -0.3792773187160492, 0.7644023895263672, -1.6848660707473755, -1.3647129535675049, 0.2210647612810135, -1.0473843812942505, 0.33695220947265625, -0.4101746082305908, -0.7502853870391846, -1.3315696716308594, -0.09247781336307526, -0.7272410988807678, -0.6115347743034363, -0.6359034776687622, 0.18767869472503662, 0.14922964572906494, -1.712921142578125, -0.16048982739448547, 0.39873284101486206, 1.0394617319107056, 0.7832769751548767, 0.20954380929470062, -1.1424494981765747, -0.7699397802352905, 0.9145488739013672, -0.7021661996841431, 1.0262646675109863, 1.2739675045013428, -1.062175989151001, 0.4406146705150604, -0.1120731458067894, 2.0277037620544434, 0.8380459547042847, -0.5104594230651855, 0.4073084592819214, -0.9176837205886841, -0.15148252248764038, 0.20498941838741302, 0.14937132596969604, 0.3077685236930847, -0.9069403409957886, -1.853279709815979, -0.3629148006439209, -0.8923571705818176, -0.7001841068267822, 2.137336015701294, 1.1628879308700562, -0.17445115745067596, -1.4334354400634766, -1.2981958389282227, 0.3611960709095001, 1.0693488121032715, -0.2885149419307709, 0.13509966433048248, 0.2548057436943054, 1.0661932229995728, -1.8804713487625122, 0.15595228970050812, 0.4055309295654297, 0.950092077255249, 0.3961379826068878, -1.1006969213485718, -0.5805535912513733, -1.5241825580596924, 0.19320248067378998, 0.1923600286245346, 0.5118316411972046, -0.969871997833252, -1.140230655670166, 1.5255985260009766, 0.18218210339546204, 1.7460033893585205, 0.4394441545009613, -1.4617071151733398, 0.8054167628288269, 1.2351303100585938, -0.45072343945503235, -1.9246586561203003, 0.6652413606643677, -0.14035257697105408, 0.2775930166244507, -2.686511993408203, 0.45712506771087646, -2.7505156993865967, -0.8674063086509705, 0.8859013915061951, -1.824477195739746, -1.8638026714324951, -0.4497790038585663, 1.6466600894927979, 1.615652322769165, -0.6082825064659119, 0.9995227456092834, 0.5789161324501038, 0.26098164916038513, 0.3543882369995117, -0.7261088490486145, 0.800044059753418, 1.059968113899231, 2.699911117553711, 0.9151961207389832, -2.4547197818756104, -1.207101821899414, 0.33010467886924744, 0.06506912410259247, -1.8644094467163086, -0.1439666450023651, -0.3627159595489502, -0.05427870899438858, -0.1272655427455902, 0.9157146215438843, 1.4595038890838623, 0.678637683391571, -1.115296721458435, -0.8681769371032715, -0.6669449806213379, -0.06094503030180931, -0.4312148690223694, -0.3544050455093384, -0.9268618822097778, 0.9449212551116943, 0.8471816778182983, 0.9504015445709229, -0.08436178416013718, 0.45087844133377075, -0.915004551410675, -1.0156728029251099, -0.5475899577140808, 0.5811843276023865, -0.15240903198719025, -1.145458698272705, 0.08997561037540436, 0.41985270380973816, 0.16032858192920685, -0.6690400838851929, 0.06208313629031181, -0.4238949120044708, -0.32779061794281006, -0.0916929841041565, 0.7513716220855713, 1.8232887983322144, 1.55674409866333, 0.6267432570457458, -0.4015841484069824, -0.3100269138813019, -0.2670179307460785, 1.15104079246521, -2.765319347381592, 0.25780177116394043, -1.9924894571304321, -0.0038701798766851425, 1.3245450258255005, -0.9109410047531128, -2.282867670059204, 1.2971206903457642, 1.5058224201202393, 0.4985770583152771, 1.2552683353424072, -0.8024897575378418, 1.1726150512695312, 1.7006242275238037, 0.640178918838501, 0.2810641825199127, 1.2832883596420288, 1.1319475173950195, -0.7488273978233337, 1.3600265979766846, 0.7446196675300598, 0.158694788813591, 1.3037189245224, 0.925452470779419, 0.7459344863891602, -0.5078482031822205, 0.8334746360778809, -1.108514428138733, -0.1264491230249405, -1.6932612657546997, -1.7043919563293457, 0.4430561065673828, -0.08200670033693314, 0.08475439995527267, 0.591949999332428, 0.34998369216918945, -0.8863843679428101, 0.07343178242444992, -0.06707020848989487, -0.5199704170227051, -0.12454844266176224, -0.365416020154953, -0.6491088271141052, -0.33426058292388916, 0.23498226702213287, 2.063225269317627, -0.8169824481010437, 0.5219173431396484, -1.4613287448883057, 1.5900248289108276, -0.33337533473968506, 0.13887566328048706, 1.2774192094802856, 0.9876771569252014, -1.4918594360351562, 0.5551928877830505, -0.2273922711610794, 0.6191235780715942, 0.15238438546657562, 0.7360392808914185, 0.2919556498527527, -0.1398543268442154, 1.1769789457321167, -1.5440465211868286, -0.43306007981300354, -0.9555990099906921, -0.14387154579162598, -0.4299427270889282, -0.14804799854755402, -0.49043819308280945, -0.4695979952812195, -1.6440515518188477, -0.03766949847340584, 0.30830490589141846, -0.5963528156280518, -0.6569375395774841, -0.6445764899253845, 1.6790409088134766, 0.188017800450325, 0.3877396285533905, -0.30040571093559265, -0.4999450743198395, -0.8785795569419861, 1.5421278476715088, -1.7004302740097046, 0.29997026920318604, -0.024041568860411644, -0.2556222975254059, -1.566407561302185, -1.235025405883789, 0.9816647171974182, -0.6937329769134521, 0.8790978193283081, 0.669728696346283, 0.5793836712837219, 0.7614914774894714, -1.2102665901184082, -0.47405970096588135, 0.6786007881164551, -1.2623622417449951, 1.6646002531051636, -0.5596384406089783, -0.06431134045124054, -1.0251259803771973, 0.36580878496170044, -0.7338070273399353, -0.2425023466348648, -1.1387232542037964, 1.1917567253112793, -1.6326267719268799, 0.45111918449401855, 0.23707015812397003, 0.4377991259098053, -1.0206522941589355, 1.033740758895874, 0.9643871188163757, 0.15958145260810852, 0.312661737203598, -0.15735924243927002, 0.6430504322052002, -0.8211991190910339, 0.5548563003540039, 0.40632644295692444, -1.4100284576416016, -2.6240503787994385, -0.11722993105649948, -0.1615600436925888, 1.2180508375167847, 0.057596806436777115, -0.4977976083755493, -1.3597984313964844, -0.08826368302106857, -0.5838378667831421, -1.1027196645736694, -0.12041668593883514, 1.0569872856140137, 0.09276256710290909, -1.0490626096725464, 0.3272729814052582, -1.0980676412582397, -1.577677607536316, 0.49425438046455383, -0.5503509640693665, 1.1135709285736084, 0.31102254986763, -0.8064202666282654, 0.21930529177188873, 1.2955149412155151, -0.18871358036994934, -1.0611307621002197, -0.6614577174186707, -0.813060998916626, -1.1809879541397095, -1.2924846410751343, 0.42237964272499084, 1.1094558238983154, -0.3618835508823395, 0.7865822911262512, 0.9081969261169434, 1.683961033821106, -0.186348095536232, -0.3494163751602173, 0.42124611139297485, 0.0975768193602562, 0.3020370602607727, 1.2965584993362427, -0.4292709529399872, 0.21653810143470764, -0.9812393188476562, -0.1594366580247879, -0.6733092069625854, -0.7575424313545227, -0.2197147011756897, -0.4465532600879669, 0.37157630920410156, 0.8038355112075806, -1.4923624992370605, 1.0033586025238037, -2.2263569831848145, 0.24745245277881622, 0.7102939486503601, -0.6507130265235901, -0.22909966111183167, -0.7817720174789429, 0.8947509527206421, -3.1721675395965576, -0.9712059497833252, -1.9369926452636719, 0.1883966028690338, 1.2992185354232788, 1.1915773153305054, 0.6681510210037231, 0.24822638928890228, -0.7660617232322693, 1.2184128761291504, 0.6233141422271729, -0.8443057537078857, -1.279564619064331, -0.6286759376525879, 1.0818129777908325, 1.250860333442688, 1.6157954931259155, -0.7754225134849548, -0.3598564863204956, 1.3827041387557983, 0.22005514800548553, 0.76422119140625, -1.320947289466858, -0.3693920075893402, -0.015788981691002846, -0.3539161682128906, -0.4137306809425354, -1.6323400735855103, 1.6311041116714478, 0.14898529648780823, -1.4533058404922485, 1.0807535648345947, 0.07243399322032928, 0.7238220572471619, -0.18416213989257812, -0.7729653716087341, -0.7473500967025757, 0.35603129863739014, 0.7881916165351868, 0.5041235089302063, 0.6608067154884338, 1.2199158668518066, 0.30562835931777954, 1.024113416671753, 1.1294760704040527, 0.3166235387325287, -0.0500045083463192, -0.40689152479171753, -1.173532485961914, -1.526172399520874, 1.3455979824066162, 0.020961429923772812, -0.5131780505180359, 1.6143012046813965, 1.0093518495559692], "dtype": "float32"}], "e": {"__nd__": [-1.2879633903503418, 0.6150814294815063, 1.87980055809021, 0.09327344596385956, 1.760190725326538, 0.5806604623794556, 1.01107656955719, 0.5693509578704834, -1.9041818380355835, 0.9206066727638245, 2.4485490322113037, -1.1804041862487793, -0.2563278377056122, 0.6239312887191772, -3.0172882080078125, -2.9271044731140137, -0.9679467082023621, -1.750205159187317, -1.2264161109924316, 2.249507427215576, 1.3950520753860474, -1.9394934177398682, 1.1707594394683838, 1.2237954139709473, 2.824643611907959, 0.40541374683380127, 0.10189463198184967, 2.3684401512145996, -1.7940095663070679, 2.055548906326294, -0.9046998023986816, 1.3691015243530273, 0.1703958511352539, -1.178384780883789, 0.2880832552909851, 1.1438770294189453, -2.1465673446655273, 0.307439923286438, -0.048133671283721924, -0.20580947399139404, 2.3462164402008057, -0.032970547676086426, 2.085554599761963, 1.5403048992156982, -0.5842745900154114, 1.276784896850586, 0.7366752624511719, 0.35501009225845337, -0.33326709270477295, 0.5730069875717163, 2.261758804321289, 0.3146648108959198, -1.3799902200698853, 1.5356276035308838, -0.8143560290336609, -2.2813515663146973, 0.49364978075027466, -0.6793308258056641, 1.045576572418213, 2.294149398803711, -0.16130752861499786, -2.3540844917297363, -0.7166672945022583, -2.5125863552093506, -1.3727221488952637, -1.0805375576019287, 0.5320361852645874, 0.05356892943382263, 0.8663563132286072, 0.27002742886543274, 0.12625765800476074, 1.3482270240783691, -1.7230987548828125, -2.9971725940704346, 0.18829035758972168, -0.9525271058082581, -3.333601474761963, -0.7744118571281433, -0.5121790170669556, -2.502124309539795, -2.8318369388580322, 2.0935020446777344, 0.39880090951919556, 0.5473012328147888, 0.3158980906009674, -0.9041469693183899, -1.1983124017715454, 1.3216650485992432, 1.2806921005249023, 0.027970343828201294, 0.22594378888607025, 1.403256893157959, -2.475493907928467, -0.287948876619339, -1.1516268253326416, -0.036819636821746826, 1.1847279071807861, -1.0774940252304077, 0.22689229249954224, -3.177145481109619, -2.0734596252441406, 1.3641178607940674, -1.1978346109390259, 1.7325536012649536, 2.861215114593506, -0.13570207357406616, 1.026853322982788, 0.7671414613723755, 0.3697871267795563, -1.8166227340698242, 0.4001653790473938, 1.467089056968689, 0.3373846411705017, 2.189326047897339, 2.7587497234344482, 0.6426888704299927, -1.5301021337509155, 3.780360698699951, -2.183887004852295, -0.08236506581306458, 0.4266015291213989, -0.5272008180618286, 0.6832524538040161, -1.1585097312927246, -0.9946174025535583, -0.3124585747718811, -0.9039585590362549, -0.16328895092010498, -0.0965447798371315, 1.046408772468567, 0.5617700815200806, -2.4509198665618896, -1.4241633415222168, -2.12857723236084, 0.8145412802696228, 3.5577454566955566, 0.47760647535324097, 1.0756443738937378, -1.6000611782073975, -2.1282920837402344, -1.6740648746490479, -2.1441893577575684, -2.387293815612793, -0.7160265445709229, 2.1672396659851074, -1.4747498035430908, 0.2231551557779312, -2.5450844764709473, -0.5388647317886353, -0.7142247557640076, -1.9480209350585938, 0.35691341757774353, 2.188953399658203, -0.008722417056560516, -0.7628648281097412, -1.1867433786392212, -2.9832053184509277, 2.187216281890869, -0.8455517888069153, -0.7870507836341858, 2.249075174331665, 0.6202212572097778, -0.918201208114624, 0.5748484134674072, -1.8944646120071411, 0.9499711990356445, 0.18760931491851807, 1.1037662029266357, -0.3294307589530945, 2.206869125366211, 2.2583305835723877, 0.7990620732307434, -0.6649221181869507, 0.22937366366386414, 2.0869312286376953, -5.171359539031982, -1.5891324281692505, 0.8246832489967346, -0.5643109083175659, 1.8259036540985107, -0.0907713770866394, 0.167549729347229, -2.364964723587036, -2.695732593536377, -2.6801297664642334, -0.09689757227897644, -1.768202304840088, -2.698805332183838, -5.0013227462768555, -2.5905656814575195, 1.1406431198120117, 2.788947105407715, -1.0601714849472046, 0.06696116924285889, -0.9072623252868652, 3.3060412406921387, -1.2178969383239746, 0.3813173770904541, 1.0723814964294434, -1.5880532264709473, -0.3339737355709076, 0.9097095727920532, -3.046032428741455, 0.20891089737415314, 0.29530784487724304, 1.7812891006469727, 2.3290600776672363, 2.5341124534606934, -1.075049877166748, 0.6750407218933105, 0.4126230478286743, -1.5004714727401733, -0.6603043079376221, 0.5788449048995972, 1.108145833015442, -0.45127415657043457, 0.9767219424247742, 0.6140216588973999, 0.08405068516731262, 0.5828850865364075, 0.131048321723938, -0.2037997841835022, -0.23528560996055603, -1.2113198041915894, -0.6164315342903137, -1.3639090061187744, -0.35922563076019287, 0.15488970279693604, -1.1293796300888062, -1.9632244110107422, 2.7902774810791016, -0.31337791681289673, 0.5977975726127625, -0.601990282535553, -1.602677345275879, -1.2950477600097656, -1.236504077911377, -0.25808119773864746, 0.7818307876586914, 0.6789084076881409, 0.08905988931655884, 0.0836426317691803, -0.5456164479255676, -1.0631930828094482, 0.014139249920845032, 0.07717258483171463, -2.2826120853424072, 0.2372819483280182, 0.014239251613616943, 0.8679786324501038, -0.20036089420318604, -0.35654646158218384, -1.1688228845596313, 2.8096420764923096, -1.8828492164611816, -1.1164746284484863, 0.618008017539978, 3.0379791259765625, -0.6819899082183838, 0.022916197776794434, 1.2619565725326538, -2.238961696624756, -0.6950162649154663, -0.7214001417160034, -0.2608675956726074, -3.0120294094085693, 0.9448177814483643, 4.069357872009277, -0.32419463992118835, 2.153650999069214, -0.8169316649436951, 0.45266854763031006, -1.7465475797653198, 2.365938186645508, 0.4808844327926636, 1.4354627132415771, -0.943947434425354, -0.7658078670501709, -1.0464028120040894, 0.3575328588485718, 1.7683240175247192, -1.7019026279449463, -0.6868200302124023, -1.1965947151184082, 1.8174703121185303, -0.5729389786720276, -0.5791252851486206, -0.06388162076473236, -1.2215542793273926, -2.518494129180908, 0.5651367902755737, 0.17709755897521973, 0.022556692361831665, -1.7353577613830566, -0.0252377986907959, 2.8110227584838867, -1.8465337753295898, 0.9490669369697571, 1.6177053451538086, 1.6568608283996582, -1.7773699760437012, 0.644051194190979, -0.3862326145172119, 0.5460400581359863, -2.224341869354248, 0.6298224329948425, -1.1366344690322876, 1.9648525714874268, 0.505113422870636, 2.241180658340454, 0.698731541633606, -1.7311042547225952, 0.798319935798645, 1.7493971586227417, -0.10008412599563599, 1.4621435403823853, 0.9189090728759766, -2.5182714462280273, 0.1193024218082428, 0.8900983333587646, 0.48245304822921753, -1.3520002365112305, -0.39872419834136963, -1.0885438919067383, 0.37800467014312744, 0.07152245193719864, -1.4462637901306152, 1.0483418703079224, -0.15114746987819672, -1.1047147512435913, 0.8145208358764648, 0.33516091108322144, 1.318376898765564, -2.9250874519348145, 0.9875373244285583, -1.8749886751174927, -1.7747159004211426, 0.9761456251144409, 0.1790452003479004, 3.1159579753875732, 0.14012905955314636, -0.4498063325881958, 0.3501628637313843, 1.2629674673080444, 2.815927028656006, 0.02862703800201416, 4.329236030578613, -1.0901858806610107, 0.458694189786911, 2.7277002334594727, 1.4305851459503174, 0.5426514148712158, -0.4489516019821167, 0.5577855110168457, 0.2791765630245209, 0.479458212852478, 1.4783397912979126, -1.6191675662994385, -1.7671787738800049, -0.37556397914886475, -1.1942660808563232, 0.06033068150281906, -1.0089221000671387, 1.0665667057037354, 2.4514763355255127, 2.300929546356201, 0.6131455898284912, 0.44868946075439453, 0.8425509929656982, -0.5035724639892578, -1.0410339832305908, 1.948486089706421, 0.7997494339942932, 0.46344295144081116, 0.01968139410018921, 2.1655356884002686, -1.1477159261703491, -0.7427898049354553, -1.0495445728302002, -0.733110249042511, 1.0742695331573486, 1.4566380977630615, 2.0173048973083496, -3.2133378982543945, 1.437131404876709, -0.4089687466621399, -2.153055191040039, 0.7648591995239258, 0.5987146496772766, 1.7334115505218506, 2.7912278175354004, -2.9103713035583496, -0.8340178728103638, -1.2346097230911255, 0.33941584825515747, 0.9826594591140747, -1.705647587776184, 1.4358097314834595, -0.8761364221572876, -0.6912786960601807, -2.844634771347046, 2.4301111698150635, -1.8679819107055664, -1.3928589820861816, -0.794398844242096, -0.16767561435699463, 0.5731040239334106, -0.7962958812713623, 1.6553295850753784, -0.47112518548965454, 0.9281198978424072, 1.3762202262878418, -0.39113849401474, 1.3674243688583374, -1.1977050304412842, -0.08599799871444702, -0.09751304984092712, -1.244388461112976, 0.8191474080085754, 0.5594648718833923, -2.081667423248291, -0.8949378728866577, 1.3767716884613037, 1.248572587966919, -0.21769219636917114, -3.1894783973693848, 0.7031974792480469, -0.5010020732879639, -1.4249099493026733, 0.006056636571884155, 1.3526008129119873, -0.028147578239440918, 0.5729876756668091, 3.9223978519439697, 0.7745258808135986, 0.7088234424591064, -1.786670446395874, -0.5347434878349304, -0.5495388507843018, 0.06341502070426941, 0.6394900679588318, 1.6204901933670044, 2.104269504547119, 2.0690526962280273, -0.2092294692993164, -0.08791637420654297, -0.3515007197856903, 0.7749162912368774, 0.5325369834899902, -1.2492296695709229, 1.068893313407898, -1.3479783535003662, -0.20218093693256378, -1.2129164934158325, 1.7546876668930054, -1.2400295734405518, -1.538610577583313, 0.08803927898406982, -0.002131938934326172, -1.25606369972229, 2.1988770961761475, 1.1429603099822998, 0.6476998329162598, -1.1171720027923584, 3.4931039810180664, -0.6650712490081787, -0.33465003967285156, -1.1074538230895996, 0.48599013686180115, -0.9758777618408203, 2.060915946960449, 1.2458820343017578, -0.5795671343803406, -1.4445748329162598, -0.8712162971496582, 2.126012086868286, 2.615933418273926, 1.372714877128601, -0.9290579557418823, -1.2975928783416748, -0.5735493898391724, -1.2288304567337036, -2.3207573890686035, 0.3498572111129761, 0.26150187849998474, 0.7140499353408813, 0.027969852089881897, 1.4634121656417847, -0.8918974995613098, 0.7296078205108643, -1.7368266582489014, -0.8009719252586365, 0.4675177335739136, -2.3608362674713135, -1.8799655437469482, 2.525662660598755, -2.416929244995117, 0.6003424525260925, 0.42498934268951416, -0.8191924095153809, 0.005360722541809082, -2.134819746017456, 0.7395958304405212, 2.746243953704834, 0.3108493685722351, -0.603087842464447, 2.247161388397217, -0.08882848173379898, 0.34810808300971985, 1.863035798072815, -1.1230745315551758, -2.0389089584350586, 0.13289502263069153, 1.1458073854446411, 2.098982810974121, 1.6818474531173706, -1.1658878326416016, -0.1851438283920288, 3.1438093185424805, 0.07087212800979614, 0.16243618726730347, 1.4391213655471802, 1.0900146961212158, -3.1942577362060547, 0.874710202217102, -1.6996477842330933, -0.6129952669143677, -2.7060670852661133, -0.4092770516872406, -3.1569766998291016, -1.1208773851394653, 1.0842251777648926, 0.6457470059394836, -1.1455388069152832, 2.046556234359741, 0.2547822594642639, 0.1306973695755005, 0.8890056610107422, -0.3757643401622772, -1.346706748008728, 1.0567402839660645, -2.0692429542541504, -0.4524114429950714, 0.32148557901382446, -0.6647792458534241, 1.3260996341705322, -0.5068174600601196, -0.14115005731582642, -1.001441240310669, 0.04898090660572052, 1.0013505220413208, 2.5914461612701416, 0.4879263639450073, 0.354073703289032, -0.5236706733703613, -2.4517948627471924, 0.6026434302330017, -0.4134204685688019, -1.5016878843307495, -1.2109042406082153, 0.44087153673171997, 0.06349611282348633, -0.9459114670753479, 0.18341195583343506, 2.6566333770751953, -0.5663378834724426, 0.06498268246650696, -0.08621668815612793, -0.6371826529502869, -1.0100938081741333, 0.6128212809562683, -0.9233707189559937, 0.7696641683578491, 1.091379165649414, -0.9211810827255249, 1.2272629737854004, -0.05222639441490173, -0.18926459550857544, -0.5139610767364502, -0.8477679491043091, -0.07601654529571533, -0.326055645942688, 0.0076615214347839355, -0.2033875435590744, 1.9970355033874512, 0.20051568746566772, 0.8095494508743286, 0.8604754209518433, 0.8826957941055298, 0.034106165170669556, 0.7748321294784546, -2.844874858856201, 1.0487462282180786, -1.9291534423828125, 2.4255731105804443, 1.924467921257019, 3.2270255088806152, 0.3466445803642273, 1.6544544696807861, -0.40351414680480957, -1.4205946922302246, -1.5707688331604004, -0.5077682733535767, 0.2285427451133728, 1.1490734815597534, 1.3618080615997314, 1.38044273853302, 1.856994867324829, 0.6980738639831543, -0.17464236915111542, -1.3824405670166016, -0.6038044095039368, -0.7309172749519348, 0.11702665686607361, 0.4136241376399994, -1.6718543767929077, -2.4019408226013184, 0.3065900504589081, -0.2821740508079529, 0.4299454987049103, 0.449218213558197, -0.43737491965293884, 0.032287001609802246, -0.28252744674682617, -2.3658714294433594, -0.17904654145240784, -0.8010929822921753, -0.786614716053009, 0.8004950284957886, -1.448591947555542, 0.28484877943992615, 0.06635689735412598, 1.236508846282959, 1.0376160144805908, 1.9841989278793335, -1.8264093399047852, -0.7804339528083801, 1.2784086465835571, -2.426953077316284, -0.576703667640686, 3.439785957336426, -2.8292949199676514, 1.015543818473816, -0.9602985382080078, 1.2645436525344849, -1.1444789171218872, -1.4111980199813843, 1.1916871070861816, -2.306426525115967, 0.4439443349838257, 0.518287181854248, 2.1676735877990723, -0.16676506400108337, 0.3564736843109131, -0.9644107818603516, -0.6495832204818726, -2.5553131103515625, 0.8092563152313232, 0.7729470729827881, 0.689471960067749, 0.7087253928184509, -0.31747984886169434, -2.1436424255371094, 0.32919439673423767, 0.8107703924179077, 0.8930436372756958, 2.639733076095581, -1.5750811100006104, 1.9826161861419678, -4.738646507263184, 0.17001865804195404, -0.7032536268234253, 2.941910743713379, 0.5625064969062805, -1.8374686241149902, -1.0798664093017578, -2.099478244781494, 0.3815140724182129, 0.03421156108379364, 1.1677172183990479, 0.5703890323638916, -0.43881863355636597, 0.2995954751968384, -1.4416247606277466, 0.9905838370323181, 1.4926594495773315, -1.796755313873291, 0.13784396648406982, 1.373412847518921, -1.8662493228912354, -3.063986301422119, 1.1755667924880981, -0.44350606203079224, -0.17564323544502258, -2.3688769340515137, -0.12774616479873657, -3.4368066787719727, -1.3272165060043335, 1.4885220527648926, -1.7646572589874268, -3.0584001541137695, -0.7528359293937683, 2.294883966445923, 3.4864044189453125, -1.7506365776062012, 1.3380579948425293, 0.3201678693294525, 1.086501121520996, 0.8039432168006897, 0.7460680603981018, 1.3194949626922607, 0.5148547887802124, 2.161317825317383, 1.3207461833953857, -2.059776544570923, 1.1908354759216309, 2.006437301635742, -1.1085093021392822, -1.5141847133636475, -1.4749555587768555, -2.01839280128479, 0.6892672777175903, 0.4244820177555084, -0.12285315990447998, 3.1031906604766846, -0.2989322543144226, -2.2830252647399902, -0.2155587077140808, -0.4460064172744751, -0.22145304083824158, -0.49828070402145386, -0.13060425221920013, -1.0664418935775757, 0.772416353225708, -1.0487749576568604, -0.3915598392486572, 0.5265877842903137, -0.25342243909835815, -0.3698112964630127, 0.21979105472564697, -0.9025408029556274, 0.8641541004180908, 1.9240672588348389, -0.4505230784416199, 1.931849479675293, 1.953259825706482, 0.9136888980865479, -0.7809338569641113, -0.7799103856086731, -1.5355929136276245, 0.1585383117198944, 0.37649038434028625, 0.5767494440078735, 2.671755313873291, 0.9522069692611694, 1.7369623184204102, -0.3615770936012268, -0.3463535010814667, -0.26729121804237366, 1.0524494647979736, -0.277055025100708, -1.3537180423736572, -1.8152668476104736, 1.2402360439300537, 2.042057991027832, -1.2130229473114014, -0.5427466630935669, 0.22862780094146729, 0.19952630996704102, 1.0666476488113403, -0.19920146465301514, -3.1804394721984863, 2.055501699447632, 2.461974620819092, 1.3724658489227295, -0.20640695095062256, 1.4662344455718994, 0.1666209101676941, 0.9855237603187561, 2.1510508060455322, 1.4868757724761963, -1.791290044784546, 2.111021041870117, 2.419424533843994, 1.9621965885162354, 0.28128212690353394, 0.8325174450874329, -1.4503473043441772, 0.4751926064491272, -1.310491919517517, -1.4680174589157104, 0.13383135199546814, 1.5992498397827148, -1.083088994026184, 0.5693175196647644, -0.23019719123840332, 0.1456587314605713, 1.5039132833480835, 1.177018642425537, 0.23543673753738403, -1.5271503925323486, 1.7158167362213135, -0.13241273164749146, -0.44272381067276, 0.256482869386673, 1.7237061262130737, -0.39310187101364136, 0.832085132598877, -1.8214287757873535, -0.979866623878479, -0.3684966564178467, 0.6157116293907166, 1.8035870790481567, 0.8342121839523315, -1.0385446548461914, 1.5835118293762207, -0.6976425647735596, 2.085402488708496, -1.8027739524841309, -0.525009274482727, -0.6131892800331116, -2.5843210220336914, 0.17721635103225708, -0.29567456245422363, 0.8084690570831299, -0.85824054479599, 1.5771304368972778, -0.19045250117778778, -0.645859956741333, -0.24880743026733398, -0.6067445874214172, -0.9081180691719055, -1.5407202243804932, 1.3077198266983032, -1.0558795928955078, 0.19838547706604004, -2.1856167316436768, 1.259782314300537, 0.2660810351371765, 1.2891921997070312, -1.5179368257522583, 1.1674991846084595, -0.39591607451438904, 1.0163524150848389, -0.4831281900405884, 0.40268129110336304, 1.1342554092407227, 2.0675253868103027, -0.797879159450531, -1.761704921722412, 0.5559573173522949, -0.11117863655090332, 1.1670032739639282, -0.1633012890815735, -0.08872586488723755, 1.4272735118865967, 1.1364316940307617, -3.104426860809326, 1.4025168418884277, -1.4734094142913818, 2.443964958190918, 0.3080829381942749, -0.011473257094621658, -2.0410995483398438, -1.031243085861206, -2.607985734939575, 0.979843258857727, -1.3484256267547607, 0.6514817476272583, -1.9103461503982544, -0.8883485794067383, 0.5416847467422485, 1.7565006017684937, 0.11780834197998047, -0.5320583581924438, 2.3278989791870117, -0.604655385017395, -1.6002905368804932, -1.386831521987915, -0.32346367835998535, -0.8281469345092773, -0.47711920738220215, -1.1039190292358398, -1.6745225191116333, -0.7001464366912842, -0.8195807933807373, 0.3026934862136841, 1.2979977130889893, 2.7156646251678467, -2.3960556983947754, -0.26441681385040283, -1.3696485757827759, -1.176253080368042, -1.1994013786315918, 3.065669536590576, 0.29398196935653687, -1.7961214780807495, 0.24830102920532227, -0.5119631290435791, -1.3410179615020752, -1.6017193794250488, 0.5912431478500366, -0.7204896211624146, 1.3813273906707764, 0.6983821392059326, 0.04750257730484009, -0.20792783796787262, 0.5467848181724548, -1.2522637844085693, -0.4684218764305115, -0.7077558636665344, -0.2598564028739929, -0.6427916884422302, 0.39339232444763184, 0.6298847198486328, 1.8618271350860596, -1.5644733905792236, -0.2937193512916565, 0.596459150314331, 0.4672433137893677, 0.5032451152801514, -1.682469129562378, 0.4896291196346283, -2.36051869392395, -0.8027426600456238, 0.9575231671333313, -0.6883106231689453, -1.5751076936721802, -0.8725702166557312, 0.8154273629188538, -0.025068819522857666, -1.4161255359649658, -0.8070115447044373, -1.6328535079956055, 1.0013642311096191, 2.2454943656921387, -2.018317937850952, -0.6944594383239746, -2.026352643966675, 0.9552218914031982, -2.0118298530578613, -0.5123569965362549, 1.935287356376648, 0.038875579833984375, 1.4452097415924072, -2.7700462341308594, 0.09069287776947021, -2.878159999847412, -0.6569411754608154, 1.7067878246307373, 0.5062479972839355, 0.9268534183502197, -1.664336085319519, -0.00810706615447998, 2.0653843879699707, 0.6669920682907104, -0.679882824420929, 0.3410865068435669, -1.7117705345153809, 0.1686563491821289, 0.15705406665802002, 1.1910653114318848, -1.7865724563598633, -0.32280799746513367, 0.7319673299789429, -1.2290940284729004, -0.7859565019607544, -0.10929012298583984, -0.43827080726623535, -0.24312680959701538, -0.3273465931415558, -0.4960927963256836, -0.8762271404266357, 2.1933631896972656, -0.44072243571281433, -2.2768328189849854, -0.005967259407043457, -1.1786973476409912, 2.1259677410125732, -1.1007013320922852, -2.792011260986328, -1.590834617614746, -0.4323303699493408, 1.1180431842803955, -0.38185906410217285, 1.1955597400665283, 1.608442783355713, 1.5724577903747559, 3.1911065578460693, 3.9783802032470703, 1.667458176612854, -1.2097047567367554, -0.8110909461975098, -0.925886332988739, 0.5419549942016602, 2.0222182273864746, 1.2243337631225586, -0.6871299147605896, 2.204585552215576, 1.7275440692901611], "dtype": "float32"}}, {"a": [{"__nd__": [1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30, 1.0000000150474662e+30], "dtype": "float32"}, {"__nd__": [-1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30, -1.0000000150474662e+30], "dtype": "float32"}], "e": {"__nd__": [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(vector_add, json.loads(_TESTS), 'cuda', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")